# AI Job Recommendation System — Model Training
**Subjects covered:** Programming for AI (CS3703) + Machine Learning

This notebook is designed to run on **Google Colab**. It:
1. Loads the dataset from a **CSV you upload directly** (no Kaggle account/API needed)
2. Cleans and engineers features (TF-IDF on skills, encodes education)
3. Trains a **from-scratch logistic regression / linear regression baseline**
   (NumPy implementation, per CS3703 W10/W11 "from scratch" requirement)
4. Trains library models (scikit-learn RandomForest) for comparison
5. Tracks every run with **MLflow**
6. Evaluates with confusion matrix / precision / recall / F1 / MAE / RMSE
7. **Saves the trained models to disk** so they never need retraining —
   the Tkinter desktop app (`app/main_app.py`) loads these saved files directly.

> Run cells top to bottom once. After Section 8, copy the `models/` folder
> next to your VS Code project (or sync the whole repo) and you're done.

## 1. Setup — clone/mount project and install dependencies

In [ ]:
# If running in Colab with this repo cloned/uploaded as a zip,
# make sure your working directory is the project root (where config.json lives).
# Example if using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/job-recommender-PAI-ML-project

!pip install -q mlflow scikit-learn pandas numpy matplotlib seaborn joblib


## 2. Upload the dataset (.zip or .csv)

You already downloaded **AI-Powered Resume Screening Dataset (2025)** from
Kaggle, so no Kaggle account or API token is needed here. Run the cell below
and use the file picker to upload either:

- the **`.zip`** file exactly as downloaded from Kaggle, **or**
- the **`.csv`** file directly (if you already unzipped it yourself)

The cell auto-detects which one you uploaded. If it's a zip, it extracts the
CSV inside automatically. Either way, the dataset ends up at
`data/raw/resume_data.csv`, which is the path the rest of the notebook (and
`config.json`) expects.

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

os.makedirs("data/raw", exist_ok=True)

print("Select the dataset file you downloaded from Kaggle (.zip or .csv).")
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
target_path = "data/raw/resume_data.csv"

if uploaded_filename.lower().endswith(".zip"):
    print(f"'{uploaded_filename}' is a zip file — extracting...")
    extract_dir = "data/raw/_extracted"
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(uploaded_filename, "r") as zf:
        zf.extractall(extract_dir)

    # Find the first CSV inside the extracted contents (handles CSVs nested
    # in subfolders too, which Kaggle zips sometimes do)
    csv_files = []
    for root, _, filenames in os.walk(extract_dir):
        for fname in filenames:
            if fname.lower().endswith(".csv"):
                csv_files.append(os.path.join(root, fname))

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file found inside the uploaded zip. "
            "Please check the zip contents and try again."
        )

    shutil.move(csv_files[0], target_path)
    shutil.rmtree(extract_dir)
    os.remove(uploaded_filename)

elif uploaded_filename.lower().endswith(".csv"):
    print(f"'{uploaded_filename}' is a CSV file — using it directly.")
    shutil.move(uploaded_filename, target_path)

else:
    raise ValueError(
        f"Unsupported file type: '{uploaded_filename}'. "
        "Please upload either the Kaggle .zip or the extracted .csv."
    )

raw_csv_path = target_path
print("Dataset ready at:", raw_csv_path)


## 3. Load & inspect raw data

In [ ]:
import pandas as pd
from src import preprocess

df_raw = preprocess.load_raw_data()
print(df_raw.shape)
df_raw.head()


In [ ]:
df_raw.info()
df_raw.describe(include='all').T


## 4. Clean data and save processed CSV (data/processed/)

In [ ]:
df_clean = preprocess.clean_dataframe(df_raw)
preprocess.save_processed_data(df_clean)
print("Cleaned shape:", df_clean.shape)
df_clean.head()


## 5. Exploratory Data Analysis

Quick look at class balance (job roles) and salary distribution, since both
feed directly into model design decisions below.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean['job_role'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title("Job Role Distribution")
axes[0].set_ylabel("Count")

sns.histplot(df_clean['salary_expectation'], bins=30, ax=axes[1])
axes[1].set_title("Salary Expectation Distribution")

plt.tight_layout()
plt.savefig("results/eda_overview.png", dpi=150)
plt.show()


## 6. Feature Engineering

TF-IDF over the skills text + numeric features (experience, projects) +
encoded education level. The fitted vectorizer/encoders are saved to
`models/` so the exact same transform can be reused at inference time
without ever needing the training data again.

In [ ]:
from src import features
import numpy as np

np.random.seed(42)  # CONFIG["random_seed"] — reproducibility (CLO2)

skills_text = features.build_skills_text_column(df_clean)
vectorizer = features.fit_skill_vectorizer(skills_text)
encoders = features.fit_label_encoders(df_clean)

X = features.build_feature_matrix(df_clean, vectorizer, encoders)
y_class = encoders['job_role'].transform(df_clean['job_role'])
y_reg = df_clean['salary_expectation'].to_numpy()

print("Feature matrix shape:", X.shape)
print("Number of job role classes:", len(encoders['job_role'].classes_))


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

print("Train:", X_train.shape, " Test:", X_test.shape)


## 7. From-Scratch Baseline — Logistic Regression (NumPy)

Per the course outline's "implement from scratch, then compare with library"
pattern (W10/W11), we implement a simple multinomial logistic regression
using only NumPy and gradient descent, then compare it to scikit-learn's
implementation below.

In [ ]:
class SoftmaxRegressionScratch:
    """Multinomial logistic regression trained with batch gradient descent,
    implemented from scratch using only NumPy (CS3703 'from scratch' requirement)."""

    def __init__(self, n_classes, lr=0.1, epochs=300, reg_lambda=0.001):
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.W = None
        self.b = None
        self.train_loss_history = []
        self.val_loss_history = []

    @staticmethod
    def _softmax(z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _one_hot(self, y):
        oh = np.zeros((y.size, self.n_classes))
        oh[np.arange(y.size), y] = 1
        return oh

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples, n_features = X.shape
        self.W = np.zeros((n_features, self.n_classes))
        self.b = np.zeros(self.n_classes)
        y_oh = self._one_hot(y)

        for epoch in range(self.epochs):
            logits = X @ self.W + self.b
            probs = self._softmax(logits)

            grad_logits = (probs - y_oh) / n_samples
            grad_W = X.T @ grad_logits + self.reg_lambda * self.W
            grad_b = grad_logits.sum(axis=0)

            self.W -= self.lr * grad_W
            self.b -= self.lr * grad_b

            train_loss = -np.mean(np.sum(y_oh * np.log(probs + 1e-9), axis=1))
            self.train_loss_history.append(train_loss)

            if X_val is not None:
                val_probs = self._softmax(X_val @ self.W + self.b)
                val_oh = self._one_hot(y_val)
                val_loss = -np.mean(np.sum(val_oh * np.log(val_probs + 1e-9), axis=1))
                self.val_loss_history.append(val_loss)

        return self

    def predict(self, X):
        probs = self._softmax(X @ self.W + self.b)
        return np.argmax(probs, axis=1)


n_classes = len(encoders['job_role'].classes_)
scratch_clf = SoftmaxRegressionScratch(n_classes=n_classes, lr=0.5, epochs=300)
scratch_clf.fit(X_train, yc_train, X_test, yc_test)

scratch_preds = scratch_clf.predict(X_test)
from sklearn.metrics import accuracy_score
print("From-scratch softmax regression accuracy:", accuracy_score(yc_test, scratch_preds))


In [ ]:
from src import evaluate

evaluate.plot_loss_curve(scratch_clf.train_loss_history, scratch_clf.val_loss_history,
                          model_name="scratch_softmax_classifier")
print("Saved training curve to results/scratch_softmax_classifier_training_curve.png")


## 8. Library Model — RandomForestClassifier (job role prediction)

This is the model that gets saved and deployed. Compared against the
from-scratch baseline above per the outline's "scratch vs library" pattern.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("ai_job_recommender")

with mlflow.start_run(run_name="job_role_classifier_rf"):
    clf = RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=4,
        class_weight="balanced", random_state=42
    )
    clf.fit(X_train, yc_train)
    preds = clf.predict(X_test)

    metrics = evaluate.evaluate_classifier(
        yc_test, preds, class_names=encoders['job_role'].classes_,
        model_name="job_role_classifier"
    )
    evaluate.save_metrics_json(metrics, "job_role_classifier")

    mlflow.log_params({"n_estimators": 200, "max_depth": 20, "model": "RandomForestClassifier"})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(clf, "job_role_classifier")

    print(metrics)


In [ ]:
# Quick baseline comparison logged separately for the report
with mlflow.start_run(run_name="job_role_classifier_logreg_baseline"):
    baseline_clf = LogisticRegression(max_iter=1000)
    baseline_clf.fit(X_train, yc_train)
    baseline_preds = baseline_clf.predict(X_test)

    baseline_metrics = evaluate.evaluate_classifier(
        yc_test, baseline_preds, class_names=encoders['job_role'].classes_,
        model_name="job_role_classifier_logreg_baseline"
    )
    mlflow.log_params({"model": "LogisticRegression"})
    mlflow.log_metrics(baseline_metrics)
    print(baseline_metrics)

print("\nFrom-scratch vs scikit-learn vs library-baseline accuracy comparison:")
print("  From-scratch softmax :", accuracy_score(yc_test, scratch_preds))
print("  sklearn RandomForest :", metrics['accuracy'])
print("  sklearn LogisticReg  :", baseline_metrics['accuracy'])


## 9. Salary Regression Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="salary_regressor_rf"):
    reg = RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=4, random_state=42
    )
    reg.fit(X_train, yr_train)
    reg_preds = reg.predict(X_test)

    reg_metrics = evaluate.evaluate_regressor(yr_test, reg_preds, model_name="salary_regressor")
    evaluate.save_metrics_json(reg_metrics, "salary_regressor")

    mlflow.log_params({"n_estimators": 200, "max_depth": 15, "model": "RandomForestRegressor"})
    mlflow.log_metrics(reg_metrics)
    mlflow.sklearn.log_model(reg, "salary_regressor")

    print(reg_metrics)


In [ ]:
with mlflow.start_run(run_name="salary_regressor_linear_baseline"):
    lin_reg = LinearRegression()
    lin_reg.fit(X_train, yr_train)
    lin_preds = lin_reg.predict(X_test)

    lin_metrics = evaluate.evaluate_regressor(yr_test, lin_preds, model_name="salary_regressor_linear_baseline")
    mlflow.log_params({"model": "LinearRegression"})
    mlflow.log_metrics(lin_metrics)
    print(lin_metrics)

print("\nRandomForest vs LinearRegression MAE comparison:")
print("  RandomForestRegressor:", reg_metrics['mae'])
print("  LinearRegression     :", lin_metrics['mae'])


## 10. Hyperparameter Tuning (GridSearchCV)

Per outline W15 requirement. Tunes the deployed RandomForestClassifier.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
}

grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid, cv=3, scoring="f1_weighted", n_jobs=-1
)
grid.fit(X_train, yc_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_clf = grid.best_estimator_
best_preds = best_clf.predict(X_test)
best_metrics = evaluate.evaluate_classifier(
    yc_test, best_preds, class_names=encoders['job_role'].classes_,
    model_name="job_role_classifier_tuned"
)
print(best_metrics)


## 11. SAVE FINAL MODELS — train once, never retrain

This is the critical cell: it persists the tuned classifier, the regressor,
the TF-IDF vectorizer, and the label encoders to `models/`. The Tkinter
desktop app loads these files directly and never calls `.fit()` again.

In [ ]:
import joblib
import json

with open("config.json") as f:
    cfg = json.load(f)

joblib.dump(best_clf, cfg["paths"]["classifier_path"])
joblib.dump(reg, cfg["paths"]["regressor_path"])
features.save_feature_artifacts(vectorizer, encoders)

print("Saved:")
print(" -", cfg["paths"]["classifier_path"])
print(" -", cfg["paths"]["regressor_path"])
print(" -", cfg["paths"]["vectorizer_path"])
print(" -", cfg["paths"]["encoder_path"])
print("\nDownload the models/ folder (and copy it next to the VS Code project) before closing this Colab session.")


## 12. (Optional) Download models folder as a zip for transfer to VS Code

In [ ]:
import shutil
shutil.make_archive("models_export", "zip", "models")

from google.colab import files
files.download("models_export.zip")
